# Compare Methods with Official T2I-CompBench Evaluation (Kaggle)

Attach the four generation notebook outputs as Kaggle datasets, then run this notebook. It copies the generated images into `/kaggle/working`, loads or clones the official T2I-CompBench suite, stages images in the official filename layout, runs the official metrics, and writes JSON/CSV/Markdown reports.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import shlex
import subprocess
import sys
import time

SEED = 13
RUN_SLUG = "t2i_compbench_seed13"
METHODS = ["spfc", "rectified_cfgpp", "cfg", "base"]
CATEGORIES = ["color", "shape", "texture", "spatial"]
EXECUTE_OFFICIAL = True
INSTALL_AIM_FLOW_DEPS = True
INSTALL_MINIMAL_EVAL_DEPS = True
INSTALL_DETECTRON2_FOR_SPATIAL = "spatial" in CATEGORIES
DOWNLOAD_SPATIAL_WEIGHTS = "spatial" in CATEGORIES

WORK_ROOT = Path("/kaggle/working") if Path("/kaggle").exists() else Path.cwd()
OUTPUT_ROOT = WORK_ROOT / RUN_SLUG
RUN_ROOT = OUTPUT_ROOT / "runs"
EVAL_DIR = OUTPUT_ROOT / "official_t2i_eval"
REPORT_DIR = OUTPUT_ROOT / "reports"
T2I_REPO_DIR = WORK_ROOT / "T2I-CompBench"
T2I_COMPBENCH_COMMIT = "1b7094991a57f3c22abdd4f6e8ba6c1a15517073"
MANIFEST_REL = Path("configs/t2i_compbench_100_seed13.json")

# Optional exact method folders or dataset roots. Leave blank to auto-discover under /kaggle/input.
METHOD_INPUTS = {
    "spfc": "",
    "rectified_cfgpp": "",
    "cfg": "",
    "base": "",
}


def run_args(args: list[str], cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    print("$", shlex.join([str(arg) for arg in args]))
    started = time.time()
    result = subprocess.run([str(arg) for arg in args], cwd=str(cwd) if cwd else None, text=True)
    print(f"elapsed: {(time.time() - started) / 60:.2f} min")
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result


def has_aim_flow_repo(path: Path) -> bool:
    return (path / "src" / "aim_flow").exists() and (path / "scripts" / "bench_evaluate.py").exists()


def find_aim_flow_repo() -> Path | None:
    cwd = Path.cwd()
    if has_aim_flow_repo(cwd):
        return cwd
    dest = WORK_ROOT / "aim-flow"
    if has_aim_flow_repo(dest):
        return dest
    input_root = Path("/kaggle/input")
    candidates = [input_root / "aim-flow", input_root / "aim-flow" / "aim-flow"]
    if input_root.exists():
        for child in sorted(input_root.glob("*")):
            candidates.extend([child, child / "aim-flow"])
    for candidate in candidates:
        if has_aim_flow_repo(candidate):
            return candidate
    return None


def ensure_working_repo() -> Path:
    source = find_aim_flow_repo()
    if source is None:
        raise FileNotFoundError("Could not find aim-flow. Attach the repo as a Kaggle dataset or run from the repo root.")
    dest = WORK_ROOT / "aim-flow" if Path("/kaggle").exists() else source
    if source.resolve() != dest.resolve():
        shutil.copytree(source, dest, dirs_exist_ok=True)
        return dest
    return source


REPO_DIR = ensure_working_repo()
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))
print("repo:", REPO_DIR)
print("run root:", RUN_ROOT)

if INSTALL_AIM_FLOW_DEPS:
    run_args([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], cwd=REPO_DIR)
    run_args([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=REPO_DIR)

In [ ]:
def method_dir_has_outputs(path: Path, method: str) -> bool:
    return path.name == method and (path / "index.json").exists() and len(list(path.glob("*.png"))) >= 100


def resolve_method_source(method: str) -> Path:
    configured = METHOD_INPUTS.get(method, "").strip()
    candidates: list[Path] = []
    if configured:
        root = Path(configured)
        candidates.extend([root, root / "t2i_compbench" / method, root / RUN_SLUG / "runs" / "t2i_compbench" / method])
    input_root = Path("/kaggle/input")
    if input_root.exists():
        candidates.extend(path.parent for path in input_root.rglob(f"t2i_compbench/{method}/index.json"))
        candidates.extend(path for path in input_root.rglob(method) if path.is_dir() and path.name == method)
    local_candidate = RUN_ROOT / "t2i_compbench" / method
    candidates.append(local_candidate)

    seen = set()
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except Exception:
            resolved = candidate
        if resolved in seen:
            continue
        seen.add(resolved)
        if method_dir_has_outputs(candidate, method):
            return candidate
    raise FileNotFoundError(f"Could not find generated outputs for {method}. Attach its Kaggle output dataset or set METHOD_INPUTS[{method!r}].")


for method in METHODS:
    source = resolve_method_source(method)
    target = RUN_ROOT / "t2i_compbench" / method
    if source.resolve() != target.resolve():
        shutil.copytree(source, target, dirs_exist_ok=True)
    images = sorted(target.glob("*.png"))
    print(f"{method}: {len(images)} images from {source}")
    assert len(images) == 100, f"Expected 100 images for {method}, found {len(images)}"

In [ ]:
def has_t2i_compbench_repo(path: Path) -> bool:
    return (path / "BLIPvqa_eval" / "BLIP_vqa.py").exists() and (path / "UniDet_eval" / "2D_spatial_eval.py").exists()


def find_t2i_compbench_repo() -> Path | None:
    if has_t2i_compbench_repo(T2I_REPO_DIR):
        return T2I_REPO_DIR
    input_root = Path("/kaggle/input")
    candidates = [input_root / "t2i-compbench", input_root / "T2I-CompBench"]
    if input_root.exists():
        for child in sorted(input_root.glob("*")):
            candidates.extend([child, child / "T2I-CompBench", child / "t2i-compbench"])
    for candidate in candidates:
        if has_t2i_compbench_repo(candidate):
            return candidate
    return None


source = find_t2i_compbench_repo()
if source is None:
    run_args(["git", "clone", "https://github.com/Karine-Huang/T2I-CompBench.git", T2I_REPO_DIR])
elif source.resolve() != T2I_REPO_DIR.resolve():
    shutil.copytree(source, T2I_REPO_DIR, dirs_exist_ok=True)

if (T2I_REPO_DIR / ".git").exists():
    run_args(["git", "-C", T2I_REPO_DIR, "fetch", "origin", T2I_COMPBENCH_COMMIT], check=False)
    run_args(["git", "-C", T2I_REPO_DIR, "checkout", T2I_COMPBENCH_COMMIT], check=False)

print("official T2I-CompBench repo:", T2I_REPO_DIR)

## Evaluation Dependencies

The official suite is older and its full `requirements.txt` pins Torch and Diffusers. This notebook installs a smaller evaluation-oriented set by default and attempts Detectron2 only when spatial evaluation is enabled.

In [ ]:
if INSTALL_MINIMAL_EVAL_DEPS:
    run_args([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "spacy>=3.5,<3.8",
        "timm==0.4.12",
        "fairscale>=0.4.4",
        "opencv-python",
        "pycocotools",
        "ruamel.yaml",
        "yacs",
        "fvcore",
        "iopath",
        "gdown",
    ])
    run_args([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=False)

if INSTALL_DETECTRON2_FOR_SPATIAL:
    run_args([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "git+https://github.com/facebookresearch/detectron2.git@5aeb252b194b93dc2879b4ac34bc51a31b5aee13",
    ], check=False)

if DOWNLOAD_SPATIAL_WEIGHTS:
    weights_dir = T2I_REPO_DIR / "UniDet_eval" / "experts" / "expert_weights"
    weights_dir.mkdir(parents=True, exist_ok=True)
    downloads = [
        (
            "https://huggingface.co/shikunl/prismer/resolve/main/expert_weights/Unified_learned_OCIM_RS200_6x%2B2x.pth",
            weights_dir / "Unified_learned_OCIM_RS200_6x+2x.pth",
        ),
        (
            "https://huggingface.co/lllyasviel/ControlNet/resolve/main/annotator/ckpts/dpt_hybrid-midas-501f0c75.pt",
            weights_dir / "dpt_hybrid-midas-501f0c75.pt",
        ),
    ]
    for url, target in downloads:
        if not target.exists():
            run_args(["curl", "-L", url, "-o", target])
    google_target = weights_dir / "Unified_learned_OCIM_RS200_6x+2x.pth"
    if not google_target.exists():
        run_args(["gdown", "https://docs.google.com/uc?id=1C4sgkirmgMumKXXiLOPmCKNTZAc3oVbq", "-O", google_target], check=False)

In [ ]:
manifest_path = REPO_DIR / MANIFEST_REL
cmd = [
    sys.executable,
    "scripts/bench_evaluate.py",
    "--benchmark",
    "t2i_compbench",
    "--manifest",
    manifest_path,
    "--run-root",
    RUN_ROOT,
    "--methods",
    *METHODS,
    "--output-dir",
    EVAL_DIR,
    "--t2i-repo-dir",
    T2I_REPO_DIR,
    "--t2i-categories",
    *CATEGORIES,
]
if EXECUTE_OFFICIAL:
    cmd.append("--execute-official")
run_args(cmd, cwd=REPO_DIR)

In [ ]:
scores_path = EVAL_DIR / "t2i_compbench_scores.json"
report_cmd = [
    sys.executable,
    "scripts/bench_report.py",
    "--t2i-scores",
    scores_path,
    "--output-dir",
    REPORT_DIR,
    "--run-root",
    RUN_ROOT,
    "--qualitative-manifest",
    REPO_DIR / MANIFEST_REL,
    "--qualitative-output",
    REPORT_DIR / "qualitative_grid.png",
    "--qualitative-methods",
    *METHODS,
    "--qualitative-max-prompts",
    "8",
]
run_args(report_cmd, cwd=REPO_DIR)

with scores_path.open("r", encoding="utf-8") as f:
    scores = json.load(f)
print(json.dumps(scores.get("scores", {}), indent=2))
print("score json:", scores_path)
print("markdown table:", REPORT_DIR / "t2i_compbench_table.md")
print("qualitative grid:", REPORT_DIR / "qualitative_grid.png")